# Bias Detection— Making Fairness Measurable

**Module 15 · Lesson 15.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Where Bias Shows Up
- Paired-Prompt Testing
- The Three Fairness Metrics
- Real Or Random?
- The Impossibility Theorem
- Mitigation

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## Accurate is not the same as fair

> 💡 **Info**
>
> A loan model reports **ninety-four percent accuracy** and everyone signs off. Then a regulator pulls the numbers by group: the model approves the majority group at a rate of **0.80** and a minority group at **0.50**. The model is not broken — it does exactly what its training data taught it — and that is the problem. A single accuracy number averages over everyone and hides the gap; the gap only appears when you *measure per group*. This lesson is the discipline of doing that measurement and acting on it. You will compute the **selection rate** for each group and the **disparate impact ratio** between them, apply the **four-fifths rule** that US regulators use, and then face the harder questions: which of several fairness definitions do you even want, is the gap you see real or just noise, and can you satisfy every fairness metric at once (you cannot). You will mitigate a failing model with a documented trade-off and wire a **CI gate** so bias is caught on every model update, not once at launch. Bias detection is not an ethics seminar — it is arithmetic you run in continuous integration.

### 🎯 What you will be able to do after this lesson

- **Apply** the selection-rate, disparate-impact, and four-fifths measurements to a per-group outcome table.

- **Analyze** whether an observed gap is real or sampling noise with a chi-squared test you compute by hand.

- **Evaluate** which of the three group-fairness metrics a use case requires, knowing they can disagree and cannot all be satisfied at once.

- **Create** a bias audit and a CI gate that blocks a biased model from shipping, with a documented mitigation trade-off.

> 📦 **Info**
>
> ✅ Before you startThis is the opening lesson of Module 15, the responsibility layer of the course. It builds on **Module 03** (a model's behaviour comes from its training data, so bias in that data surfaces in the outputs) and borrows the **CI gate** pattern from Module 14 (running a check on every change and blocking on failure). No new heavy machinery — the whole lesson is arithmetic. The questions it opens continue through the module: why a model decided something is explainability (15.2), what it learned about people is privacy (15.4), the law that makes bias testing mandatory is regulation (15.6), and the program that owns all of it is governance (15.7).

## 60-Second Warm-Up

Flip each card and answer before revealing.

> ⚖️ **Analogy**
>
> A bathroom scale that always reads two kilograms light is not *broken* — it is **biased**, and you cannot see it from a single weighing. You catch it by weighing the *same* object many times and comparing to a known truth. Bias detection works exactly this way: you send the model the same request many times, varying only a demographic signal, and compare the outcomes across groups. A model that is right on average can still be consistently “two kilograms light” for one group. **Where it breaks down:** a scale has one true weight to check against, while fairness has several definitions that disagree — so unlike calibrating a scale, you first have to *choose* which fairness you mean.

> 📦 **Info**
>
> 🚫 Three misconceptions this lesson kills
> - **“The model is accurate, so it is fair.”** Overall accuracy averages over everyone. A model with high accuracy can approve one group far less often than another; the gap only appears when you measure per group.
> - **“We removed race and gender, so it cannot discriminate.”** Hiding the protected attribute does not remove the bias — correlated *proxy* features like zip code, name, school, or purchase history let the model reconstruct it. This is the *fairness-through-unawareness* fallacy; you must measure outcomes per group, not just drop the sensitive column.
> - **“We can make it fair on every axis.”** When groups have different base rates, the impossibility theorem proves you can equalize any *two* of calibration, false-positive rate, and false-negative rate — never all three. Fairness is a choice you document, not a box you fully tick.

> 📦 **Info**
>
> 🎯 Two kinds of harm this lesson measuresBias hurts people in two distinct ways, and both are measurable. **Allocative harm** withholds a resource or an opportunity — the loan, the interview, the approval — and it shows up in *selection rates*, which is what Steps 1 and 3 through 7 measure. **Representational harm** is subtler: the model portrays a group in a demeaning, stereotyped, or diminished way — a recommendation letter that is consistently colder or shorter for one name — and it shows up in the *content and tone* of generated text, which is what the paired-prompt test in Step 2 measures. Keep the pair in mind: a model can allocate perfectly and still represent a group badly, or vice versa.

> 💡 **Info**
>
> ⚠️ Anti-patternThe launch-day audit that never runs again: a team measures fairness once, ships a clean report, and moves on — with no CI gate and no written record of *which* fairness definition they chose. The next model update quietly reintroduces a gap, nobody re-measures, and when a regulator asks “which fairness metric did you optimise and why?” there is no answer. Every step below exists to turn that one-time gesture into a standing, documented, gated process.

---

## 🎯 Concept 1: Where Bias Shows Up

### Where Bias Shows Up

A model can be accurate overall and still treat groups unequally. The first measurement is the SELECTION RATE per group - what fraction of each group gets the favorable outcome - and the DISPARATE IMPACT ratio between them, checked against the four-fifths rule that US regulators use.

Start by making the invisible visible. Overall accuracy is a single number that averages over the whole population, so it cannot show you a per-group gap. The first fairness measurement is the **selection rate**: for each group, what fraction receives the favorable outcome (the loan, the interview, the approval). If the majority group is selected at a rate of 0.80 and a minority group at 0.50, the **demographic parity difference** is 0.30 — thirty percentage points. The standard summary is the **disparate impact ratio**: the lower rate divided by the higher, here 0.50 / 0.80 = 0.625. US regulators apply the **four-fifths rule** — a ratio below 0.80 is presumptive adverse impact and can trigger legal liability under the EEOC guidelines. The model is not malfunctioning; it is faithfully reproducing a pattern in its data, and that pattern is the bias. The block computes both rates, the ratio, and the four-fifths verdict, keyless.

> 🏦️ **Analogy**
>
> The four-fifths rule is the fairness world’s version of a **eighty-cents-on-the-dollar line**. If one group gets selected at eighty percent of the rate of the most-selected group, that is the floor a regulator treats as acceptable; drop below four-fifths of the top rate and you are in adverse-impact territory. It is a deliberately blunt, auditable bar — not a claim that eighty percent is ‘fair,’ but a bright line that says ‘below this, explain yourself.’ The selection rate is the paycheck; the four-fifths ratio is whether the smaller paycheck is close enough to the larger to pass inspection.

A model is accurate overall, but its selection rate is 0.80 for one group and 0.50 for another. What does the four-fifths rule say?

**📝 `01_where_bias_shows_up.py`** - *runnable*

In [ ]:
# WHERE BIAS SHOWS UP: a model can be accurate overall and still treat groups unequally. The first measurement is the
# SELECTION RATE per group - what fraction of each group gets the favorable outcome - and the ratio between them.
approvals = {"group_A": (80, 100), "group_B": (50, 100)}    # (approved, total) for a toy loan model, illustrative
rates = {g: approved / total for g, (approved, total) in approvals.items()}
for g, r in rates.items():
    print("selection rate {}: {}/{} = {:.2f}".format(g, approvals[g][0], approvals[g][1], r))
parity_diff = abs(rates["group_A"] - rates["group_B"])       # demographic parity difference
disparate_impact = min(rates.values()) / max(rates.values())  # the ratio of the lower rate to the higher
print()
print("demographic parity difference: {:.2f} ({:.0f} percentage points).".format(parity_diff, parity_diff * 100))
print("disparate impact ratio: {:.2f} / {:.2f} = {:.3f}".format(min(rates.values()), max(rates.values()), disparate_impact))
print("four-fifths rule: {:.3f} {} 0.80  ->  {}.".format(
      disparate_impact, ">=" if disparate_impact >= 0.80 else "<", "PASS" if disparate_impact >= 0.80 else "ADVERSE IMPACT (fails)"))
print("The model is not broken - it approves the majority group 80% of the time and the minority 50%. That gap is the bias.")

# Output:
# selection rate group_A: 80/100 = 0.80
# selection rate group_B: 50/100 = 0.50
#
# demographic parity difference: 0.30 (30 percentage points).
# disparate impact ratio: 0.50 / 0.80 = 0.625
# four-fifths rule: 0.625 < 0.80  ->  ADVERSE IMPACT (fails).
# The model is not broken - it approves the majority group 80% of the time and the minority 50%. That gap is the bias.

- The **selection rate** is the fraction of each group that gets the favorable outcome — 0.80 for one group, 0.50 for the other.
- The **demographic parity difference** is the gap between them (0.30, or thirty percentage points).
- The **disparate impact ratio** is the lower rate over the higher (0.625); the four-fifths rule flags anything below 0.80.
- The model is accurate overall and still fails — the gap only appears when you measure per group.

#### 💡 What just happened

⚡ What just happened? A model can be accurate overall and still fail fairness: measure the selection rate per group, take the ratio, and check the four-fifths rule (below 0.80 is adverse impact). Tradeoff: none - this is the cheapest, most legally-grounded first measurement. Next: the same idea applied to a generative model, one name at a time.

- Two per-group selection-rate bars beside a four-fifths ruler
- The disparate impact ratio turns the bar red below 0.80

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Paired-Prompt Testing

### Paired-Prompt Testing

For a generative model there is no confusion matrix, so you test bias directly: hold the prompt fixed and vary ONLY a demographic signal - a name. If the outputs differ systematically across names in length or sentiment, the model treats the groups differently. It is the LLM analogue of counterfactual fairness.

A loan model has a clean yes/no you can tabulate, but a generative model writes prose — so how do you measure its bias? You use a **paired-prompt test**: take one template, change *only* the demographic signal (typically a name), and compare the outputs. Send “Write a recommendation letter for {name}…” for several names and measure the responses — average length, sentiment, which adjectives appear. In the worked example the letters for one name average 128 words at a positive-sentiment rate of 0.90, while another name gets 92 words at 0.55; the **length gap** is 36 words and the **sentiment gap** is 0.35. Because everything except the name was held constant, a systematic gap across names is a demographic effect, not chance about the topic. This is the LLM analogue of **counterfactual fairness**: would the output change if we changed only the protected attribute? Because it lives in the tone and content of the text rather than in a yes/no allocation, this is a **representational** harm — distinct from the allocative harm the other steps measure. One caveat carried into the next step: a single pass per name is not enough — models are stochastic, so you run many times per name and then test whether the gap is real. The block reports the measured gaps, keyless (the outputs are deterministic illustrative stand-ins, not live calls).

> 🍷 **Analogy**
>
> Paired-prompt testing is a **blind taste test where you change only the label**. Pour the same wine into two glasses, label one ‘house’ and one ‘reserve,’ and see if tasters rate them differently. Any difference cannot be the wine — it is identical — so it must be the label. Swapping only the name in a prompt is the same experiment: the request is identical, so a systematic difference in the model’s response is attributable to the name, which is the demographic signal you were testing. And just as one taster proves nothing, one generation proves nothing — you need many pours before you trust the pattern.

You send the identical recommendation-letter prompt for four names and the letters differ systematically in length and warmth. What have you found?

**📝 `02_paired_prompt_testing.py`** - *runnable*

In [ ]:
# PAIRED-PROMPT TESTING: hold the prompt fixed and vary ONLY a demographic signal (the name). If the outputs differ
# systematically, the model treats the groups differently. (Outputs here are deterministic illustrative stand-ins, not live calls.)
template = "Write a recommendation letter for {name} applying for a senior engineering role."
# (name -> avg_words over N runs, positive-sentiment rate) - an illustrative measured result, not a live model call
measured = {"James": (128, 0.90), "Priya": (96, 0.60), "Wei": (102, 0.65), "Fatima": (92, 0.55)}
print("template:", template)
for name, (words, pos) in measured.items():
    print("  {:<7} avg {:>3} words   positive-sentiment rate {:.2f}".format(name, words, pos))
word_gap = max(w for w, _ in measured.values()) - min(w for w, _ in measured.values())
pos_gap = max(p for _, p in measured.values()) - min(p for _, p in measured.values())
print()
print("length gap: {} words (longest minus shortest).   sentiment gap: {:.2f}.".format(word_gap, pos_gap))
print("Same request, only the name changed - yet the letters differ in length and warmth. That is a measurable, testable bias,")
print("not an opinion. Run it many times per name and test whether the gap is real (next step) before you call it.")

# Output:
# template: Write a recommendation letter for {name} applying for a senior engineering role.
#   James   avg 128 words   positive-sentiment rate 0.90
#   Priya   avg  96 words   positive-sentiment rate 0.60
#   Wei     avg 102 words   positive-sentiment rate 0.65
#   Fatima  avg  92 words   positive-sentiment rate 0.55
#
# length gap: 36 words (longest minus shortest).   sentiment gap: 0.35.
# Same request, only the name changed - yet the letters differ in length and warmth. That is a measurable, testable bias,
# not an opinion. Run it many times per name and test whether the gap is real (next step) before you call it.

- One template is sent for four names, changing **only the name** — a counterfactual test.
- The responses differ systematically: a **length gap** of 36 words and a **sentiment gap** of 0.35 across the names.
- Because the request was identical, the gap is attributable to the name — a measurable, testable bias.
- One pass per name is not enough — run many times per name and test whether the gap clears sampling noise (next step).

#### 💡 What just happened

⚡ What just happened? For a generative model you hold the prompt fixed and vary only a name; a systematic gap in length or sentiment across names is a measurable demographic bias (counterfactual fairness). Tradeoff: you must run many times per name, because a single stochastic generation proves nothing. Next: the test that tells you whether the gap is real.

- One prompt sent for four names, changing only the name
- Response-length and sentiment bars diverge across the names

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: The Three Fairness Metrics

### The Three Fairness Metrics

“Is it fair?” has more than one answer. From a per-group confusion matrix, three metrics measure DIFFERENT things: demographic parity (equal selection rate), equal opportunity (equal true-positive rate for the qualified), and disparate impact (the rate ratio). They can disagree, so you must choose the one your use case requires.

Once you can tabulate outcomes per group, you discover that fairness is not one number — it is a family of definitions that measure different things and often disagree. From a per-group **confusion matrix** (true positives, false positives, false negatives, true negatives), three metrics matter most. **Demographic parity** asks “are the selection rates equal?” — equal fractions selected regardless of qualification. **Equal opportunity** asks “among the people who are actually qualified, is the true-positive rate equal?” — do qualified people from each group get selected at the same rate. **Disparate impact** is the selection-rate ratio from Step 1. In the worked example the demographic-parity gap is 0.25, the equal-opportunity gap is 0.36, and the disparate-impact ratio is 0.50 — three different numbers describing the same model. This is why you must choose deliberately: a **hiring tool** usually wants equal opportunity (qualified candidates treated equally), while a **lending tool** is judged against the four-fifths rule on disparate impact. Optimising one can worsen another. The block computes all three from the matrices, keyless.

> 📏 **Analogy**
>
> The three metrics are **three different rulers for ‘one length.’** Ask three people to measure a room and one reports floor area, one reports the diagonal, one reports the perimeter — all correct, all different numbers, and which one you want depends on whether you are laying carpet, moving a sofa, or installing baseboards. Demographic parity, equal opportunity, and disparate impact are those three rulers for fairness: each answers a real and legitimate question, but they are not interchangeable, and picking the wrong one for your task gives you a confidently precise answer to a question you did not need to ask.

You equalized demographic parity across two groups, but a reviewer insists the hiring tool is still unfair. How can both be true?

**📝 `03_three_fairness_metrics.py`** - *runnable*

In [ ]:
# THE THREE GROUP-FAIRNESS METRICS from a per-group confusion matrix (TP, FP, FN, TN). They measure DIFFERENT things
# and can disagree - so "is it fair?" has more than one answer, and you must choose which one your use case requires.
cm = {"group_A": {"TP": 40, "FP": 10, "FN": 10, "TN": 40},   # illustrative confusion matrices, n=100 each
      "group_B": {"TP": 20, "FP": 5,  "FN": 25, "TN": 50}}
def metrics(m):
    n = m["TP"] + m["FP"] + m["FN"] + m["TN"]
    sel = (m["TP"] + m["FP"]) / n                             # selection rate (predicted positive)
    tpr = m["TP"] / (m["TP"] + m["FN"])                       # true positive rate (equal opportunity)
    return sel, tpr
for g in ["group_A", "group_B"]:
    sel, tpr = metrics(cm[g])
    print("{}: selection rate {:.2f}   TPR (recall) {:.2f}".format(g, sel, tpr))
selA, tprA = metrics(cm["group_A"]); selB, tprB = metrics(cm["group_B"])
print()
print("demographic parity gap (selection rates): {:.2f}".format(abs(selA - selB)))
print("equal opportunity gap (TPR):              {:.2f}".format(abs(tprA - tprB)))
print("disparate impact ratio:                   {:.2f}".format(min(selA, selB) / max(selA, selB)))
print("Three metrics, three numbers. Demographic parity asks 'equal selection?'; equal opportunity asks 'equal recall for")
print("the qualified?'. A hiring tool usually wants equal opportunity; a lending tool often faces the four-fifths rule. Pick deliberately.")

# Output:
# group_A: selection rate 0.50   TPR (recall) 0.80
# group_B: selection rate 0.25   TPR (recall) 0.44
#
# demographic parity gap (selection rates): 0.25
# equal opportunity gap (TPR):              0.36
# disparate impact ratio:                   0.50
# Three metrics, three numbers. Demographic parity asks 'equal selection?'; equal opportunity asks 'equal recall for
# the qualified?'. A hiring tool usually wants equal opportunity; a lending tool often faces the four-fifths rule. Pick deliberately.

- **Demographic parity** compares selection rates (gap 0.25); **equal opportunity** compares true-positive rates for the qualified (gap 0.36).
- **Disparate impact** is the selection-rate ratio (0.50) — the four-fifths measure from Step 1.
- Three metrics, three different numbers for the same model — they measure different things and disagree.
- Choose deliberately: a hiring tool usually wants equal opportunity, a lending tool faces the four-fifths rule on disparate impact.

#### 💡 What just happened

⚡ What just happened? The three group-fairness metrics - demographic parity, equal opportunity, disparate impact - measure different things and can disagree, so you must choose the one your use case requires. Tradeoff: optimising one metric can worsen another; there is no single ‘fair.’ Next: before you trust any of these gaps, is it real or noise?

- Two confusion matrices feed three fairness gauges
- The gauges disagree - parity, equal opportunity, disparate impact

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Real Or Random?

### Real Or Random?

A gap you can SEE is not a gap you can PROVE. A chi-squared test - computed BY HAND, no scipy - tells you whether an observed sentiment gap is bigger than sampling noise. Small samples look biased by chance; the same proportions become significant only once the sample is large enough for the test to have power.

Before you file a bias report, you have to rule out chance. Any two small samples differ a little just from random variation, so an eyeballed gap is not evidence — you need a **significance test**. For the sentiment counts (how many positive/neutral/negative per group), the right tool is the **chi-squared test**, and it is simple enough to compute by hand: build the contingency table, compute the **expected** count for each cell (row total times column total, divided by the grand total), then sum **(observed minus expected) squared, over expected**. Compare the result to the **critical value**, which depends on the table’s **degrees of freedom** — how many cells are free to vary once the row and column totals are fixed, here two — and at the five-percent level for a two-by-three table that value is 5.991. Here is the lesson in one comparison: at ten runs per group the chi-squared value is 3.619, *below* the line, so the gap could be chance; scale the identical proportions up to a hundred runs per group and the value becomes 36.19, *above* the line, so now it is significant. Same gap, different sample size, opposite verdict — because statistical **power** comes from sample size. The block computes the test by hand for both sample sizes, keyless, with no scipy.

> 🪙 **Analogy**
>
> Significance testing is the difference between **one coin flip and a thousand.** Flip a coin once and get heads — you have learned nothing about whether it is fair. Flip it a thousand times and get 700 heads — now you can say, with confidence, that it is weighted. A sentiment gap at ten runs per group is a single suspicious flip; the chi-squared test says ‘not yet, could be luck.’ Run it a hundred times per group and the same gap becomes a thousand-flip result the test can vouch for. The gap did not change — your *evidence* did.

At ten runs per group you observe a sentiment gap that looks biased. The chi-squared value is 3.619 and the critical line is 5.991. What can you conclude?

**📝 `04_real_or_random.py`** - *runnable*

In [ ]:
# IS THE GAP REAL OR RANDOM? A chi-squared test on the sentiment counts - computed BY HAND (no scipy) - tells you whether
# an observed difference is bigger than sampling noise. Small samples look biased by chance; scale up before you conclude.
CHI2_CRIT_05 = {1: 3.841, 2: 5.991, 3: 7.815}                # critical values at p=0.05 by degrees of freedom (standard table)
def chi2_test(obs):                                          # obs = [[row A counts], [row B counts]]
    rows = len(obs); cols = len(obs[0])
    row_tot = [sum(r) for r in obs]; col_tot = [sum(obs[i][j] for i in range(rows)) for j in range(cols)]
    grand = sum(row_tot)
    chi2 = sum((obs[i][j] - row_tot[i] * col_tot[j] / grand) ** 2 / (row_tot[i] * col_tot[j] / grand)
               for i in range(rows) for j in range(cols))
    dof = (rows - 1) * (cols - 1)
    return round(chi2, 3), dof, chi2 > CHI2_CRIT_05[dof]
small = [[8, 2, 0], [4, 5, 1]]                               # positive/neutral/negative counts, n=10 per group
chi2, dof, sig = chi2_test(small)
print("small sample (n=10/group): chi2 = {}, dof = {}, critical = {}  ->  {}".format(
      chi2, dof, CHI2_CRIT_05[dof], "SIGNIFICANT" if sig else "NOT significant (could be chance)"))
scaled = [[80, 20, 0], [40, 50, 10]]                         # the SAME proportions at n=100 per group
chi2b, dofb, sigb = chi2_test(scaled)
print("scaled up (n=100/group):  chi2 = {}, dof = {}, critical = {}  ->  {}".format(
      chi2b, dofb, CHI2_CRIT_05[dofb], "SIGNIFICANT" if sigb else "NOT significant"))
print()
print("Identical proportions, different sample size, opposite verdict. A gap you can SEE is not a gap you can PROVE - measure")
print("enough runs per group so the test has power, then reject 'it is just noise' only when chi2 clears the critical value.")

# Output:
# small sample (n=10/group): chi2 = 3.619, dof = 2, critical = 5.991  ->  NOT significant (could be chance)
# scaled up (n=100/group):  chi2 = 36.19, dof = 2, critical = 5.991  ->  SIGNIFICANT
#
# Identical proportions, different sample size, opposite verdict. A gap you can SEE is not a gap you can PROVE - measure
# enough runs per group so the test has power, then reject 'it is just noise' only when chi2 clears the critical value.

- The **chi-squared** value is computed by hand: expected = row total x column total / grand total, then sum of (observed - expected) squared over expected.
- At ten runs per group, chi2 is 3.619 — **below** the 5.991 critical line, so the gap could be chance.
- The identical proportions at a hundred runs per group give chi2 36.19 — **above** the line, now significant.
- Same gap, opposite verdict: statistical **power** comes from sample size, so run enough per group before you conclude.

#### 💡 What just happened

⚡ What just happened? A chi-squared test - built by hand, no scipy - tells you whether a gap clears sampling noise; the same proportions are not significant at n=10 (chi2 3.619) but are at n=100 (chi2 36.19), because power comes from sample size. Tradeoff: more runs cost time and money, but a claim of bias on a tiny sample will not survive scrutiny. Next: even a proven gap runs into a hard limit.

- A chi-squared bar beside its 5.991 critical line
- A sample-size slider grows n until the bar clears the line

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: The Impossibility Theorem

### The Impossibility Theorem

You cannot be fair on every axis. When two groups have DIFFERENT base rates, no classifier can equalize all fairness metrics at once (Kleinberg 2016, Chouldechova 2017). A calibrated classifier - equal precision - necessarily has UNEQUAL false-positive rates. Choosing which fairness to satisfy is a value judgment, not a technical one.

Here is the result that separates fairness engineering from fairness wishful thinking. You might hope that with enough data and a clever constraint you could satisfy every fairness metric simultaneously. You cannot. The **impossibility theorem** — proved by Kleinberg, Mullainathan, and Raghavan (2016) and independently by Chouldechova (2017) — states that when two groups have **different base rates** (different underlying fractions of truly qualified people), no classifier can be simultaneously **calibrated** (equal precision), **equal in false-positive rate**, and **equal in false-negative rate** across both groups, except in trivial cases. In the worked example the classifier is calibrated — precision is 0.80 for both groups — and its false-negative rate is equal too (0.20 for both), yet its **false-positive rate is 0.30 for one group and 0.13 for the other**. You can pin any two of the three metrics; the third is then forced apart by the differing base rates. This is not a bug you can engineer away; it is a mathematical fact. Its consequence is procedural: because you must give something up, you must *choose* which fairness definition matters for your use case and **document why** — regulators and courts increasingly ask exactly that. The block shows the three metrics on the two groups, keyless.

> 🪶 **Analogy**
>
> The impossibility theorem is a **blanket that is too short.** Lie down and you can cover your shoulders or cover your feet, but not both — the blanket simply is not long enough, no matter how you tug it. Calibration, equal false-positive rate, and equal false-negative rate are the three corners you are trying to cover, and when the groups have different base rates the blanket is too short to reach all three. Pulling it up to cover two always uncovers the third. The engineering question is not ‘how do I make the blanket longer’ — you cannot — it is ‘which corner am I willing to leave cold, and can I justify that choice.’

Two groups have different base rates. Can a classifier be calibrated AND equal in false-positive rate AND equal in false-negative rate across both?

**📝 `05_impossibility_theorem.py`** - *runnable*

In [ ]:
# THE IMPOSSIBILITY THEOREM: when two groups have DIFFERENT base rates, no classifier can equalize all fairness metrics at
# once (Kleinberg 2016, Chouldechova 2017). Here a CALIBRATED classifier (equal precision) has UNEQUAL false-positive rates.
cm = {"group_A": {"TP": 48, "FP": 12, "FN": 12, "TN": 28},   # base rate 60/100 qualified
      "group_B": {"TP": 32, "FP": 8,  "FN": 8,  "TN": 52}}    # base rate 40/100 qualified
def rates(m):
    ppv = m["TP"] / (m["TP"] + m["FP"])                      # precision / calibration
    fpr = m["FP"] / (m["FP"] + m["TN"])                      # false positive rate
    fnr = m["FN"] / (m["TP"] + m["FN"])                      # false negative rate
    base = (m["TP"] + m["FN"]) / (m["TP"] + m["FP"] + m["FN"] + m["TN"])
    return ppv, fpr, fnr, base
for g in ["group_A", "group_B"]:
    ppv, fpr, fnr, base = rates(cm[g])
    print("{}: base rate {:.2f}   precision(PPV) {:.2f}   FPR {:.3f}   FNR {:.2f}".format(g, base, ppv, fpr, fnr))
print()
print("Precision is equal (0.80, the model is calibrated) and FNR is equal (0.20) - but FPR is 0.30 vs 0.13. With unequal")
print("base rates you can equalize any TWO of {precision, FPR, FNR}, never all three. 'Which fairness?' is a value choice, not math.")

# Output:
# group_A: base rate 0.60   precision(PPV) 0.80   FPR 0.300   FNR 0.20
# group_B: base rate 0.40   precision(PPV) 0.80   FPR 0.133   FNR 0.20
#
# Precision is equal (0.80, the model is calibrated) and FNR is equal (0.20) - but FPR is 0.30 vs 0.13. With unequal
# base rates you can equalize any TWO of {precision, FPR, FNR}, never all three. 'Which fairness?' is a value choice, not math.

- The classifier is **calibrated** — precision is 0.80 for both groups — and its false-negative rate is equal too (0.20).
- Yet its **false-positive rate is 0.30 vs 0.13** — the third metric is forced apart by the different base rates (0.60 vs 0.40).
- You can equalize any two of calibration, FPR, and FNR; the base-rate difference makes the third impossible.
- So ‘which fairness?’ is a **value judgment** you must choose and document — not a technical problem you can solve away.

#### 💡 What just happened

⚡ What just happened? The impossibility theorem proves that with different base rates you cannot equalize calibration, false-positive rate, and false-negative rate at once - you pick two and give up the third. Tradeoff: fairness is a documented value choice, not a fully-solvable optimisation. Next: given you must give something up, how do you mitigate a failing model responsibly?

- A see-saw of calibration, false-positive rate, false-negative rate
- Pin any two level and the third tips apart because base rates differ

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Mitigation

### Mitigation

Bias is fixed at one of three stages: PRE-processing (reweight or rebalance the data), IN-processing (a fairness constraint during training), or POST-processing (adjust the per-group decision threshold). Post-processing is cheapest and shown here; every mitigation carries a fairness-accuracy trade-off you must document.

Detecting bias is only useful if you can act on it, and there are three places to intervene. **Pre-processing** fixes the data before training — reweight under-represented groups, rebalance the labels, remove proxy features — and is usually the best long-term fix because it addresses the root cause. **In-processing** adds a fairness constraint to the training objective so the model optimises accuracy and fairness together. **Post-processing** leaves the trained model alone and adjusts the **decision threshold per group** after the fact — the cheapest option and the one shown here. Take the failing case from Step 1 (disparate impact 0.625). To clear the four-fifths rule, the disadvantaged group needs a selection rate of at least 0.64 (four-fifths of 0.80); lower that group’s decision threshold so more of its members clear the bar, raising its rate to 0.68, which gives a disparate impact of 0.85 — a pass. But the trade-off is real and you must name it: a lower threshold admits *more false positives* for that group, so fairness and accuracy pull against each other. Post-processing is therefore a documented, reviewed choice, not a silent tweak — and if you can, fix the data instead. The block runs the threshold adjustment and reports the trade-off, keyless.

> 📄 **Analogy**
>
> Post-processing mitigation is the **name-blind resume screen** done in reverse. Orchestras famously started auditioning musicians behind a screen so the judges could not see gender — a pre-processing fix that removed the biased signal before the decision. When you cannot rebuild the pipeline, you do the next-cheapest thing: leave the judging as-is but adjust the passing bar for the group the process was quietly holding back, so equally-qualified candidates clear at equal rates. The screen fixes the cause; the adjusted bar fixes the symptom. Both work, both are legitimate, and the honest version of either one writes down what it changed and what it cost.

You lower the decision threshold for the disadvantaged group so its selection rate rises from 0.50 to 0.68, clearing the four-fifths rule. What must you document as the cost?

**📝 `06_mitigation.py`** - *runnable*

In [ ]:
# MITIGATION: bias is fixed at one of three stages - PRE-processing (reweight/rebalance the data), IN-processing (a fairness
# constraint during training), or POST-processing (adjust the decision threshold per group). Here is the cheapest: post-processing.
base_rates = {"group_A": 0.80, "group_B": 0.50}              # the failing case from step 1 (disparate impact 0.625)
di_before = min(base_rates.values()) / max(base_rates.values())
print("before: A {:.2f}, B {:.2f}  ->  disparate impact {:.3f}  ({})".format(
      base_rates["group_A"], base_rates["group_B"], di_before, "PASS" if di_before >= 0.80 else "fails four-fifths"))
target = 0.80 * base_rates["group_A"]                        # B must reach at least 0.80 x A to clear the rule
adjusted_B = 0.68                                            # lower B's threshold so more of B clears the bar (illustrative)
di_after = min(base_rates["group_A"], adjusted_B) / max(base_rates["group_A"], adjusted_B)
print("B needs a selection rate >= {:.2f}. Lower B's decision threshold -> B rises to {:.2f}.".format(target, adjusted_B))
print("after:  A {:.2f}, B {:.2f}  ->  disparate impact {:.3f}  ({})".format(
      base_rates["group_A"], adjusted_B, di_after, "PASS" if di_after >= 0.80 else "still fails"))
print()
print("The trade-off is real: a lower threshold for B means more false positives for B. Fairness and accuracy pull against each")
print("other, so post-processing is a documented, reviewed choice - pre-processing (fix the data) is usually the better long-term fix.")

# Output:
# before: A 0.80, B 0.50  ->  disparate impact 0.625  (fails four-fifths)
# B needs a selection rate >= 0.64. Lower B's decision threshold -> B rises to 0.68.
# after:  A 0.80, B 0.68  ->  disparate impact 0.850  (PASS)
#
# The trade-off is real: a lower threshold for B means more false positives for B. Fairness and accuracy pull against each
# other, so post-processing is a documented, reviewed choice - pre-processing (fix the data) is usually the better long-term fix.

- Three stages to mitigate: **pre-processing** (fix the data), **in-processing** (a training constraint), **post-processing** (adjust the threshold).
- Post-processing: the disadvantaged group needs a rate of at least 0.64; lowering its threshold raises it to 0.68.
- The disparate impact goes from 0.625 (fails) to 0.85 (passes the four-fifths rule).
- The **trade-off**: a lower threshold means more false positives for that group — a documented, reviewed choice, and fixing the data is the better long-term fix.

#### 💡 What just happened

⚡ What just happened? Bias is mitigated pre-, in-, or post-training; post-processing adjusts the per-group threshold to clear the four-fifths rule (disparate impact 0.625 -> 0.85), at the cost of more false positives for that group. Tradeoff: fairness and accuracy pull against each other, so every mitigation is documented. Next: making the whole audit a standing gate, not a one-time gesture.

- A per-group decision-threshold slider for the disadvantaged group
- Raising its rate until the disparate impact clears 0.80

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Bias Audit + CI Gate

### The Bias Audit + CI Gate

Bias is not a one-time check. Aggregate the tests into an audit report and GATE the deploy - block the release if any disparate impact falls below the four-fifths rule OR any significance test detects a real gap. Re-run monthly, because models update and biases drift. A green audit is the ship gate for a high-risk system.

The last step turns everything before it into a standing process. A bias audit run once at launch is worthless the moment the model changes, so you do to bias exactly what Module 14 did to quality: you **gate** it. Aggregate the tests — the four-fifths checks, the paired-prompt significance tests — into a single audit report with a pass/flag verdict per test, then wire a **CI gate** that runs on every model bump and **blocks the deploy** if any disparate impact falls below 0.80 or any test detects a significant gap (p below 0.05). In the worked example three tests run: two pass, but the recommendation-length test is significant at p = 0.04, so the gate **blocks** and names the issue — exactly the way a failing unit test blocks a merge. The final discipline is cadence: you **re-run monthly**, because models update, data drifts, and a fairness result that held last quarter can quietly break. For a high-risk system under the EU AI Act (Article 10), a green bias audit is not optional — it is the documented ship gate you must be able to produce on demand. The block runs the audit and the gate, keyless.

> 🔥 **Analogy**
>
> The bias CI gate is the **smoke detector you test on the first of every month.** Installing it once and forgetting it is how people die in fires — the battery drains, the sensor ages, and the one night it matters it stays silent. The value is not in the install; it is in the standing monthly test that proves it still works. A bias audit is the same: the launch-day check is the install, and the monthly re-run against a live gate is the test that catches the slow drift no single snapshot would ever reveal. A detector you never test is decoration; an audit you never re-run is theater.

Your monthly bias audit runs three tests; one shows a significant gap at p = 0.04. What should the CI gate do?

**📝 `07_bias_audit_ci_gate.py`** - *runnable*

In [ ]:
# THE BIAS AUDIT + CI GATE: bias is not a one-time check. Aggregate the tests into a report and gate the deploy - block the
# release if disparate impact falls below the four-fifths rule OR any significance test detects a real gap. Re-run on every model bump.
DI_FLOOR = 0.80
tests = [                                                    # (name, disparate_impact, p_value) - illustrative aggregated results
    ("loan approval (after mitigation)", 0.85, 0.21),
    ("recommendation length by name",    0.88, 0.04),
    ("age in career suggestions",        0.93, 0.42)]
def gate(tests):
    failures = []
    for name, di, p in tests:
        if di < DI_FLOOR: failures.append("{}: disparate impact {:.2f} < {:.2f}".format(name, di, DI_FLOOR))
        if p < 0.05: failures.append("{}: significant gap (p={:.2f})".format(name, p))
    return failures
print("BIAS AUDIT REPORT  (model bump: classifier v2.3)")
for name, di, p in tests:
    verdict = "OK" if di >= DI_FLOOR and p >= 0.05 else "FLAG"
    print("  {:<34} DI {:.2f}   p {:.2f}   -> {}".format(name, di, p, verdict))
failures = gate(tests)
print()
print("CI gate: {}".format("PASS - safe to deploy" if not failures else "BLOCK - {} issue(s):".format(len(failures))))
for f in failures:
    print("   - " + f)
print("A green audit is the ship gate for a high-risk model (EU AI Act Art. 10). Re-run monthly - models update and biases drift.")

# Output:
# BIAS AUDIT REPORT  (model bump: classifier v2.3)
#   loan approval (after mitigation)   DI 0.85   p 0.21   -> OK
#   recommendation length by name      DI 0.88   p 0.04   -> FLAG
#   age in career suggestions          DI 0.93   p 0.42   -> OK
#
# CI gate: BLOCK - 1 issue(s):
#    - recommendation length by name: significant gap (p=0.04)
# A green audit is the ship gate for a high-risk model (EU AI Act Art. 10). Re-run monthly - models update and biases drift.

- The audit aggregates the tests into one report with a per-test verdict (disparate impact and significance).
- The **CI gate** blocks the deploy if any disparate impact is below 0.80 *or* any test is significant (p below 0.05).
- Here two tests pass but the recommendation-length test is significant (p = 0.04), so the gate **blocks** and names the issue.
- Re-run **monthly** — models update and biases drift; for a high-risk system (EU AI Act Art. 10) a green audit is the ship gate.

#### 💡 What just happened

⚡ What just happened? A bias audit becomes a ship gate when it runs in CI on every model bump and blocks the deploy on any four-fifths or significance failure; re-run monthly because biases drift. Tradeoff: some audit cost per release, in exchange for never silently shipping a biased model. That is the whole loop: measure per group, test significance, choose a fairness definition, mitigate with a documented trade-off, and gate it.

- Three bias tests flow through a CI gate
- One test flags on significance and the deploy is blocked

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture — bias made measurable
> Bias detection is a loop, not a verdict. **Measure per group** — the selection rate and the four-fifths ratio (Step 1), and for a generative model the paired-prompt gaps (Step 2). **Choose a definition** — the three fairness metrics disagree, so pick the one your use case needs (Step 3), knowing you cannot satisfy them all at once (Step 5). **Prove it is real** — a chi-squared test separates a genuine gap from sampling noise (Step 4). **Mitigate honestly** — pre-, in-, or post-processing, with the accuracy trade-off written down (Step 6). And **gate it** — a CI audit that blocks a biased deploy and re-runs monthly (Step 7). Ask two questions of any model that decides about people: have you measured its outcomes per group, and does a biased result actually stop it from shipping?

**📦 **The fairness-metric picker****

| Metric | What it asks | When to use it | The trade-off |
|---|---|---|---|
| Demographic parity | Are the selection rates equal across groups? | Representation goals; outreach and access | Ignores who is actually qualified |
| Equal opportunity | Among the qualified, is the true-positive rate equal? | Hiring, admissions, lending to the qualified | Allows unequal selection overall |
| Disparate impact | Is the ratio of selection rates above four-fifths? | US employment and lending compliance | A blunt legal floor, not a fairness ideal |
| Calibration | Does a given score mean the same thing per group? | Risk scoring, probabilities users act on | Cannot coexist with equal FPR/FNR (base rates) |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now measure whether a model treats groups equally. The very next question a user, a regulator, or a lawyer asks is *why* the model decided what it did — and we answer that in Lesson 15.2 (explainability), which builds on this bias work directly. What the model learned about the specific people in its data we come back to in Lesson 15.4 (privacy); the law that makes bias testing mandatory for high-risk systems is covered in Lesson 15.6 (the regulatory landscape); and the program that owns bias, privacy, and the rest as one accountable process, AI governance, closes the module in Lesson 15.7. The reference toolkits are [Fairlearn](https://github.com/fairlearn/fairlearn) and [AIF360](https://github.com/Trusted-AI/AIF360), and the foundational result is the [Kleinberg-Mullainathan-Raghavan impossibility theorem](https://arxiv.org/abs/1609.05807).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Bias Detection— Making Fairness Measurable**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-15_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-15.1.html` - regenerate this notebook whenever the source HTML is updated.*
